# 🧪 AiStock 完整流程端到端测试

## 测试目标
- ✅ 完整系统流程测试
- ✅ 数据→计算→组合→输出全流程
- ✅ 模拟每日运行流程

## 测试流程
1. 加载配置
2. 获取数据
3. 计算动态价格
4. 组合管理
5. 生成报告

In [ ]:
# 环境设置
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("="*60)
print("🚀 AiStock 完整流程端到端测试")
print(f"📅 测试时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

In [ ]:
# 导入系统模块
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from base_services.database_service import DatabaseService
from dynamic_price_system.data.data_loader import DataLoader
from dynamic_price_system.core.dynamic_price_engine import DynamicPriceEngine
from dynamic_price_system.portfolio.tracker import PortfolioTracker
from dynamic_price_system.portfolio.risk_manager import RiskManager
from dynamic_price_system.utils.export_utils import ExportUtils
from config.global_settings import DATABASE_MAIN_URL, DB_POOL_CONFIG, OUTPUT_DIR

print("✅ 系统模块导入成功")

## 步骤 1: 初始化系统

In [ ]:
print("\n【步骤 1】初始化系统...")

# 初始化服务
config = ConfigService(system_name='dynamic_price')
cache = CacheService(max_size=2000, ttl=3600)
db = DatabaseService(DATABASE_MAIN_URL, DB_POOL_CONFIG)

# 初始化数据加载器
data_loader = DataLoader(
    config_service=config,
    cache_service=cache,
    db_main=db,
    enable_cache=True
)

# 初始化计算引擎
price_engine = DynamicPriceEngine(config_service=config, cache_service=cache)

# 初始化组合管理
portfolio = PortfolioTracker(initial_capital=1000000, config=config)
risk_manager = RiskManager(config, portfolio=portfolio)

# 初始化工具
exporter = ExportUtils(output_dir=OUTPUT_DIR)

print("✅ 系统初始化完成")

## 步骤 2: 数据获取

In [ ]:
print("\n【步骤 2】数据获取...")

# 加载股票数据
stocks_data = data_loader.load_all_stocks()
print(f"✅ 股票数据：{len(stocks_data)}只")

# 加载宏观数据
macro_data = data_loader.load_all_macro()
print(f"✅ 宏观数据：{len(macro_data)}个指标")

# 加载财务数据
financial_data = data_loader.load_all_financial()
print(f"✅ 财务数据：{len(financial_data)}只")

# 缓存统计
cache_stats = data_loader.get_cache_stats()
print(f"📊 缓存命中率：{cache_stats['hit_rate']:.1%}")

## 步骤 3: 动态价格计算

In [ ]:
print("\n【步骤 3】动态价格计算...")

results = price_engine.calculate_all(
    stocks_data=stocks_data,
    financial_data=financial_data,
    macro_data=macro_data
)

print(f"✅ 计算完成：{len(results)}只标的")

# 统计建议分布
recommendations = {}
for r in results:
    rec = r['recommendation']
    recommendations[rec] = recommendations.get(rec, 0) + 1

print("\n📊 投资建议分布：")
for rec, count in sorted(recommendations.items(), key=lambda x: -x[1]):
    print(f"  • {rec}: {count}只")

In [ ]:
# 显示推荐标的
recommended = [r for r in results if r['recommendation'] in ['强烈推荐', '推荐']]
recommended = sorted(recommended, key=lambda x: -x['profit_loss_ratio'])

print(f"\n📊 推荐标的（盈亏比 Top 5）：")
for r in recommended[:5]:
    print(f"  • {r['code']} {r.get('sector', '')}")
    print(f"    入场¥{r['entry_price']:.2f} | 止损¥{r['stop_loss']:.2f} | 目标¥{r['target_price']:.2f}")
    print(f"    盈亏比{r['profit_loss_ratio']:.2f} | 评分{r['fundamental_score']}")

## 步骤 4: 组合管理

In [ ]:
print("\n【步骤 4】组合管理...")

# 模拟建仓（前 5 只推荐标的）
current_prices = {r['code']: r['current_price'] for r in results}

for r in recommended[:5]:
    qty = int(100000 / r['entry_price'] / 100) * 100  # 每只 10 万，100 股整数倍
    if qty > 0:
        portfolio.buy(r['code'], r['entry_price'], qty)

print(f"✅ 建仓完成：{len(portfolio.positions)}只标的")
print(f"📊 剩余现金：¥{portfolio.cash:,.2f}")

In [ ]:
# 风险检查
alerts = risk_manager.check_alerts(
    dynamic_prices=results,
    current_prices=current_prices,
    positions=portfolio.positions
)

print(f"\n📊 风险预警：{len(alerts)}个")
for alert in alerts[:3]:
    print(f"  [{alert['level']}] {alert['message'][:60]}...")

## 步骤 5: 生成报告

In [ ]:
print("\n【步骤 5】生成报告...")

# 导出动态价格表
price_file = exporter.export_dynamic_prices(results)
print(f"✅ 动态价格表：{price_file}")

# 导出组合摘要
summary = portfolio.get_summary(current_prices)
portfolio_file = exporter.export_portfolio_summary(summary)
print(f"✅ 组合摘要：{portfolio_file}")

# 导出风险报告
risk_report = risk_manager.generate_risk_report(alerts, summary)
risk_file = exporter.export_risk_report(risk_report)
print(f"✅ 风险报告：{risk_file}")

# 导出每日综合报告
daily_file = exporter.export_daily_report(results, summary, risk_report)
print(f"✅ 每日综合报告：{daily_file}")

## 📊 测试总结

In [ ]:
print("\n" + "="*60)
print("📋 完整流程测试总结")
print("="*60)
print(f"✅ 数据获取：{len(stocks_data)}只股票 + {len(macro_data)}个宏观指标")
print(f"✅ 价格计算：{len(results)}只标的")
print(f"✅ 组合建仓：{len(portfolio.positions)}只标的")
print(f"✅ 风险预警：{len(alerts)}个")
print(f"✅ 报告生成：4 个文件")
print(f"📊 缓存命中率：{cache_stats['hit_rate']:.1%}")
print(f"📊 组合收益：{summary['total_return']}")
print("="*60)

# 清理资源
db.close()
cache.clear()
print("\n✅ 资源已清理")
print("\n🎉 端到端测试完成！")